In [1]:
import requests
import datetime, time, logging, os, traceback
import pandas as pd
from numpy import nan

from urllib.parse import quote_plus
from sqlalchemy import create_engine
engine = create_engine('postgresql://zbrothers:%s@localhost:5434/zbrothers' % quote_plus('zbrothers@2026'))

In [2]:
API_TOKEN = os.environ['OLIST_API_TOKEN']

today = datetime.date.today()
# today = datetime.datetime(2026, 4, 15)
today_str = today.strftime('%d/%m/%Y')

In [3]:
situacao_dict = {
    'aguardando_separacao': 0,
    'em_separacao': 4,
    'separada': 2,
    'embalada': 3
}

In [4]:
def getInfosPesquisar(url: str, params: dict) -> pd.DataFrame:
    """
    Busca informações que são paginadas
    """
    res = requests.get(url, params=params)
    print(res.json())
    if res.json()['retorno']['status'] != 'OK':
        print(f"Falha ao obter quantidade de páginas: {res.json()['retorno']['status']}, possivelmente por sobrecarga na API.\nAguardando 90 segundos para tentar novamente...")
        time.sleep(90) 
        
        res = requests.get(url, params=params)
        
    try:
        # Código 32 indica que não há dados para a consulta, então retornamos um DataFrame vazio
        if (res.json()['retorno']['codigo_erro'] == 32):
            return pd.DataFrame()  # Retorna DataFrame vazio se não houver dados para a consulta
    except:
        pass
    
    num_pages = res.json()['retorno']['numero_paginas']
    print(f"Quantidade de páginas: {num_pages}")
    
    for i in range(1, num_pages + 1):
        print(f"Obtendo página {i} de {num_pages}...")
        params['pagina'] = i
        res = requests.get(url, params=params)
        
        if res.json()['retorno']['status'] != 'OK':
            print(f"Falha ao obter página {i}: {res.json()['retorno']['status']}, possivelmente por sobrecarga na API.\nAguardando 90 segundos para tentar novamente...")
            time.sleep(90) 
            
            res = requests.get(url, params=params)
        
        df = pd.DataFrame(res.json()['retorno']['separacoes'])
        
        if i == 1:
            df_final = df
        else:
            df_final = pd.concat([df_final, df], ignore_index=True)

    return df_final

In [5]:
separacoes_aguardando_separacao = getInfosPesquisar(
    url='https://api.tiny.com.br/api2/separacao.pesquisa.php',
    params={
        'token': API_TOKEN,
        'formato': 'json',
        'situacao': situacao_dict['aguardando_separacao'],
        'dataInicial': {today_str}
    }
)

{'retorno': {'status_processamento': '3', 'status': 'OK', 'separacoes': [{'id': '1075534873', 'idOrigem': '1075534868', 'objOrigem': 'notafiscal', 'situacao': '3', 'dataCriacao': '27/04/2026', 'dataSeparacao': '27/04/2026', 'dataCheckout': '27/04/2026', 'destinatario': 'Eduardo Luis Lupatini', 'idContato': '1031482939', 'numero': '187977', 'dataEmissao': '27/04/2026', 'idFormaEnvio': '675234044', 'numeroPedidoEcommerce': '2000012685051505', 'idOrigemVinc': '1075529565', 'objOrigemVinc': 'venda', 'situacaoVenda': '6'}, {'id': '1075534881', 'idOrigem': '1075534874', 'objOrigem': 'notafiscal', 'situacao': '3', 'dataCriacao': '27/04/2026', 'dataSeparacao': None, 'dataCheckout': '27/04/2026', 'destinatario': 'Laura Cibele Souza Bezerra', 'idContato': '1031482947', 'numero': '187978', 'dataEmissao': '27/04/2026', 'idFormaEnvio': '675234044', 'numeroPedidoEcommerce': '2000012685060293', 'idOrigemVinc': '1075529573', 'objOrigemVinc': 'venda', 'situacaoVenda': '6'}, {'id': '1075534890', 'idOrig

In [6]:
separacoes_em_separacao = getInfosPesquisar(
    url='https://api.tiny.com.br/api2/separacao.pesquisa.php',
    params={
        'token': API_TOKEN,
        'formato': 'json',
        'situacao': situacao_dict['em_separacao'],
        'dataInicial': {today_str}
    }
)

{'retorno': {'status_processamento': '1', 'status': 'Erro', 'codigo_erro': 32, 'erros': [{'erro': 'A consulta não retornou registros'}]}}
Falha ao obter quantidade de páginas: Erro, possivelmente por sobrecarga na API.
Aguardando 90 segundos para tentar novamente...


In [7]:
separacoes_separadas = getInfosPesquisar(
    url='https://api.tiny.com.br/api2/separacao.pesquisa.php',
    params={
        'token': API_TOKEN,
        'formato': 'json',
        'situacao': situacao_dict['separada'],
        'dataInicial': {today_str}
    }
)

{'retorno': {'status_processamento': '3', 'status': 'OK', 'separacoes': [{'id': '1075536733', 'idOrigem': '1075536649', 'objOrigem': 'notafiscal', 'situacao': '2', 'dataCriacao': '27/04/2026', 'dataSeparacao': '27/04/2026', 'dataCheckout': None, 'destinatario': 'ANNA KARLA', 'idContato': '1031480356', 'numero': '188253', 'dataEmissao': '27/04/2026', 'idFormaEnvio': '675234044', 'numeroPedidoEcommerce': '2000016123951546', 'idOrigemVinc': '1075412026', 'objOrigemVinc': 'venda', 'situacaoVenda': '6'}, {'id': '1075536980', 'idOrigem': '1075536759', 'objOrigem': 'notafiscal', 'situacao': '2', 'dataCriacao': '27/04/2026', 'dataSeparacao': '27/04/2026', 'dataCheckout': None, 'destinatario': 'DOCES MIRAHY LTDA', 'idContato': '1031480900', 'numero': '188275', 'dataEmissao': '27/04/2026', 'idFormaEnvio': '675234044', 'numeroPedidoEcommerce': '2000016129027422', 'idOrigemVinc': '1075516323', 'objOrigemVinc': 'venda', 'situacaoVenda': '6'}, {'id': '1075537085', 'idOrigem': '1075536788', 'objOrige

In [8]:
separacoes_embaladas = getInfosPesquisar(
    url='https://api.tiny.com.br/api2/separacao.pesquisa.php',
    params={
        'token': API_TOKEN,
        'formato': 'json',
        'situacao': situacao_dict['embalada'],
        'dataInicial': {today_str}
    }
)

{'retorno': {'status_processamento': '3', 'status': 'OK', 'separacoes': [{'id': '1075534873', 'idOrigem': '1075534868', 'objOrigem': 'notafiscal', 'situacao': '3', 'dataCriacao': '27/04/2026', 'dataSeparacao': '27/04/2026', 'dataCheckout': '27/04/2026', 'destinatario': 'Eduardo Luis Lupatini', 'idContato': '1031482939', 'numero': '187977', 'dataEmissao': '27/04/2026', 'idFormaEnvio': '675234044', 'numeroPedidoEcommerce': '2000012685051505', 'idOrigemVinc': '1075529565', 'objOrigemVinc': 'venda', 'situacaoVenda': '6'}, {'id': '1075534881', 'idOrigem': '1075534874', 'objOrigem': 'notafiscal', 'situacao': '3', 'dataCriacao': '27/04/2026', 'dataSeparacao': None, 'dataCheckout': '27/04/2026', 'destinatario': 'Laura Cibele Souza Bezerra', 'idContato': '1031482947', 'numero': '187978', 'dataEmissao': '27/04/2026', 'idFormaEnvio': '675234044', 'numeroPedidoEcommerce': '2000012685060293', 'idOrigemVinc': '1075529573', 'objOrigemVinc': 'venda', 'situacaoVenda': '6'}, {'id': '1075534890', 'idOrig